# Tarea 1 – Regresión lineal múltiple y regularizaciónBase de datos `Credit` (paquete `ISLR2`). Variable respuesta: `Balance`.

In [1]:
## Instalación de librerías pertinentes (solo si faltan)

paquetes <- c("ISLR2", "glmnet")
faltan <- paquetes[!paquetes %in% rownames(installed.packages())]
if (length(faltan) > 0) install.packages(faltan, repos = "https://cloud.r-project.org")

In [2]:
library(ISLR2)
library(glmnet)

Loading required package: Matrix



Loaded glmnet 5.0



In [3]:
data(Credit)

head(Credit)
dim(Credit)
str(Credit)

,Income,Limit,Rating,Cards,Age,Education,Own,Student,Married,Region,Balance
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<dbl>
1,14.891,3606,283,2,34,11,No,No,Yes,South,333
2,106.025,6645,483,3,82,15,Yes,Yes,Yes,West,903
3,104.593,7075,514,4,71,11,No,No,No,West,580
4,148.924,9504,681,3,36,11,Yes,No,No,West,964
5,55.882,4897,357,2,68,16,No,No,Yes,South,331
6,80.180,8047,569,4,77,10,No,No,No,South,1151


[1] 400  11

'data.frame':	400 obs. of  11 variables:
 $ Income   : num  14.9 106 104.6 148.9 55.9 ...
 $ Limit    : num  3606 6645 7075 9504 4897 ...
 $ Rating   : num  283 483 514 681 357 569 259 512 266 491 ...
 $ Cards    : num  2 3 4 3 2 4 2 2 5 3 ...
 $ Age      : num  34 82 71 36 68 77 37 87 66 41 ...
 $ Education: num  11 15 11 11 16 10 12 9 13 19 ...
 $ Own      : Factor w/ 2 levels "No","Yes": 1 2 1 2 1 1 2 1 2 2 ...
 $ Student  : Factor w/ 2 levels "No","Yes": 1 2 1 1 1 1 1 1 1 2 ...
 $ Married  : Factor w/ 2 levels "No","Yes": 2 2 1 1 2 1 1 1 1 2 ...
 $ Region   : Factor w/ 3 levels "East","South",..: 2 3 3 3 2 2 1 3 2 1 ...
 $ Balance  : num  333 903 580 964 331 ...


# Problema 1: Regresión lineal múltiple$$Balance_i = \beta_0 + \beta_1 Income_i + \beta_2 Limit_i + \beta_3 Cards_i + \beta_4 Age_i + \varepsilon_i$$

### 1.1 Ajuste por mínimos cuadrados y ecuación estimada

In [4]:
modelo1 <- lm(Balance ~ Income + Limit + Cards + Age, data = Credit)
summary(modelo1)


Call:
lm(formula = Balance ~ Income + Limit + Cards + Age, data = Credit)

Residuals:
    Min      1Q  Median      3Q     Max 
-230.51 -115.37  -38.75   37.91  521.33 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -3.996e+02  3.584e+01 -11.150  < 2e-16 ***
Income      -7.493e+00  3.838e-01 -19.522  < 2e-16 ***
Limit        2.628e-01  5.798e-03  45.329  < 2e-16 ***
Cards        2.142e+01  5.953e+00   3.599  0.00036 ***
Age         -8.876e-01  4.811e-01  -1.845  0.06580 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 162.7 on 395 degrees of freedom
Multiple R-squared:  0.876,	Adjusted R-squared:  0.8748 
F-statistic: 697.9 on 4 and 395 DF,  p-value: < 2.2e-16


**Ecuación estimada**$$\widehat{Balance} = -399{,}62 - 7{,}4931\,Income + 0{,}2628\,Limit + 21{,}4215\,Cards - 0{,}8876\,Age$$

### 1.2 Interpretación de los coeficientes de Income y Limit**Income (−7,4931):** manteniendo fijos `Limit`, `Cards` y `Age`, un aumento de mil dólares en el ingreso se asocia a una disminución promedio de 7,49 unidades en el saldo. El signo negativo se explica porque el efecto se mide a cupo constante: entre dos personas con el mismo límite de crédito, la de mayor ingreso tiende a mantener un saldo menor. Conviene notar que la correlación simple entre `Income` y `Balance` es positiva (0,46); el signo negativo aparece recién al controlar por `Limit`.**Limit (0,2628):** manteniendo fijos `Income`, `Cards` y `Age`, un dólar adicional de límite de crédito se asocia a un aumento promedio de 0,263 unidades en el saldo, es decir, unos 26 dólares más de saldo por cada 100 dólares adicionales de cupo. Es el predictor con el efecto más fuerte del modelo.

### 1.3 Significancia individual ($\alpha = 0{,}05$)Para cada predictor se contrasta$$H_0: \beta_j = 0 \qquad \text{vs} \qquad H_1: \beta_j \neq 0$$usando el estadístico $t_j = \hat\beta_j / EE(\hat\beta_j)$, que bajo $H_0$ sigue una distribución $t$ con $n - p - 1 = 395$ grados de libertad. Se rechaza $H_0$ cuando el valor-p es menor que 0,05.Presentan evidencia de asociación con `Balance`: **Income** (p < 2e−16), **Limit** (p < 2e−16) y **Cards** (p = 0,00036). **Age** no presenta evidencia suficiente al 5 % (p = 0,0658): no podemos descartar que su efecto sea nulo una vez controlados los demás predictores.

### 1.4 $R^2$, $R^2_{aj}$ y test F global```Residual standard error: 162.7 on 395 degrees of freedomMultiple R-squared:  0.876,	Adjusted R-squared:  0.8748F-statistic: 697.9 on 4 and 395 DF,  p-value: < 2.2e-16```El modelo explica el **87,6 %** de la variabilidad de `Balance`, con un $R^2$ ajustado de **87,48 %**. Que ambos valores sean casi idénticos indica que los cuatro predictores aportan información real y no solo grados de libertad: el $R^2_{aj}$ penaliza la inclusión de variables irrelevantes.El test F global contrasta$$H_0: \beta_1 = \beta_2 = \beta_3 = \beta_4 = 0 \qquad \text{vs} \qquad H_1: \text{al menos un } \beta_j \neq 0$$con $F = \dfrac{SSR/p}{SSE/(n-p-1)} \sim F_{p,\,n-p-1}$ bajo $H_0$.Se obtiene $F(4, 395) = 697{,}9$ con valor-p $< 2{,}2\times10^{-16}$, por lo que **se rechaza $H_0$**: existe evidencia de asociación entre el conjunto de predictores y el saldo promedio. El error estándar residual es de 162,7 unidades de `Balance`.

### Extra (matemática del código)$\hat\beta = (X'X)^{-1}X'y$, solución de las ecuaciones normales $X'X\beta = X'y$.

In [5]:
X <- cbind(Intercepto = 1, Income = Credit$Income,
           Limit = Credit$Limit, Cards = Credit$Cards, Age = Credit$Age)
y <- Credit$Balance

beta_gorro <- solve(t(X) %*% X) %*% t(X) %*% y
beta_gorro

Intercepto,-399.6206858
Income,-7.4930547
Limit,0.2628013
Cards,21.4215371
Age,-0.8876401


In [6]:
y_gorro  <- X %*% beta_gorro
residuos <- y - y_gorro

head(y_gorro)
head(residuos)

449.1250
543.7209
698.6410
1014.4566
451.0739
1131.6860


-116.12499
359.27914
-118.64105
-50.45663
-120.07388
19.31400


In [7]:
SSE <- sum(residuos^2)
SST <- sum((y - mean(y))^2)
R2  <- 1 - SSE/SST

n <- nrow(X)
p <- ncol(X) - 1
R2_ajustado <- 1 - (1 - R2) * (n - 1) / (n - p - 1)

c(R2 = R2, R2_ajustado = R2_ajustado)

R2 R2_ajustado 
  0.8760446   0.8747894

In [8]:
sigma2_gorro <- SSE / (n - p - 1)

Var_beta_gorro <- sigma2_gorro * solve(t(X) %*% X)
error_estandar <- sqrt(diag(Var_beta_gorro))

t_value <- as.vector(beta_gorro) / error_estandar
p_value <- 2 * pt(-abs(t_value), df = n - p - 1)

tabla_manual <- data.frame(
  Estimate  = as.vector(beta_gorro),
  Std_Error = error_estandar,
  t_value   = t_value,
  p_value   = p_value
)

print(tabla_manual)

               Estimate    Std_Error    t_value       p_value
Intercepto -399.6206858 35.839268713 -11.150358  2.742602e-25
Income       -7.4930547  0.383831055 -19.521752  6.698783e-60
Limit         0.2628013  0.005797683  45.328673 1.317855e-158
Cards        21.4215371  5.952548362   3.598717  3.604316e-04
Age          -0.8876401  0.481132869  -1.844896  6.580110e-02


In [9]:
SSR      <- SST - SSE
F_global <- (SSR/p) / (SSE/(n - p - 1))

p_F <- pf(F_global, df1 = p, df2 = n - p - 1, lower.tail = FALSE)

c(F_global = F_global, p_valor = p_F)

F_global       p_valor 
 6.979076e+02 1.447132e-177

**Rechazamos $H_0$: existe evidencia de que el conjunto de variables `Income`, `Limit`, `Cards` y `Age` tiene una asociación estadísticamente significativa con `Balance`.** Los valores coinciden exactamente con los entregados por `lm()`.

# Problema 2: Selección de modelos

### 2.1 Comparación con el modelo del Problema 1

In [10]:
modelo_completo <- lm(Balance ~ ., data = Credit)
summary(modelo_completo)


Call:
lm(formula = Balance ~ ., data = Credit)

Residuals:
    Min      1Q  Median      3Q     Max 
-161.64  -77.70  -13.49   53.98  318.20 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -479.20787   35.77394 -13.395  < 2e-16 ***
Income        -7.80310    0.23423 -33.314  < 2e-16 ***
Limit          0.19091    0.03278   5.824 1.21e-08 ***
Rating         1.13653    0.49089   2.315   0.0211 *  
Cards         17.72448    4.34103   4.083 5.40e-05 ***
Age           -0.61391    0.29399  -2.088   0.0374 *  
Education     -1.09886    1.59795  -0.688   0.4921    
OwnYes       -10.65325    9.91400  -1.075   0.2832    
StudentYes   425.74736   16.72258  25.459  < 2e-16 ***
MarriedYes    -8.53390   10.36287  -0.824   0.4107    
RegionSouth   10.10703   12.20992   0.828   0.4083    
RegionWest    16.80418   14.11906   1.190   0.2347    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 98.79 on 388 degrees of freedom
Mul

In [11]:
comparacion <- data.frame(
  Modelo = c("Problema 1 (4 var)", "Completo (todas)"),
  R2_aj  = c(summary(modelo1)$adj.r.squared,
             summary(modelo_completo)$adj.r.squared),
  AIC    = c(AIC(modelo1), AIC(modelo_completo))
)
comparacion

Modelo,R2_aj,AIC
<chr>,<dbl>,<dbl>
Problema 1 (4 var),0.8747894,5215.578
Completo (todas),0.9538287,4823.370


El modelo completo presenta un $R^2_{aj}$ de **0,9538** frente a **0,8748** del modelo del Problema 1, y un AIC de **4823,4** frente a **5215,6**. Como el $R^2_{aj}$ es mayor y el AIC menor, ambos criterios coinciden en preferir el modelo completo. La mejora es sustancial y no se debe únicamente al mayor número de predictores: el $R^2_{aj}$ penaliza la inclusión de variables irrelevantes, por lo que su aumento indica una ganancia real de ajuste. Una diferencia de casi 400 puntos de AIC es enorme frente a la regla habitual de que diferencias menores a 2 son irrelevantes.Buena parte de la ganancia proviene de `Student`, cuyo coeficiente estimado es **+425,7** dólares: es la variable individual con mayor efecto y estaba ausente del modelo del Problema 1.**Preferimos entonces el modelo con todas las variables.**

### Extra (matemática del código)$AIC = -2\log L + 2k$, con $k = p + 2$ (los $p$ coeficientes, el intercepto y $\sigma^2$).

In [12]:
X_all <- model.matrix(Balance ~ ., data = Credit)
n_all <- nrow(X_all)
p_all <- ncol(X_all) - 1

beta_all  <- solve(t(X_all) %*% X_all) %*% (t(X_all) %*% y)
y_hat_all <- X_all %*% beta_all
e_all     <- y - y_hat_all

SSE_all    <- sum(e_all^2)
R2_all     <- 1 - SSE_all/SST
R2_adj_all <- 1 - (1 - R2_all) * (n_all - 1) / (n_all - p_all - 1)

AIC_manual <- n_all * log(SSE_all/n_all) + 2*(p_all + 2) + n_all * (1 + log(2*pi))

c(R2_adj_all = R2_adj_all, AIC_manual = AIC_manual)

R2_adj_all   AIC_manual 
   0.9538287 4823.3703910

### 2.2 Selección backward por AIC a partir del modelo completoSe usa `trace = FALSE` para mostrar solo el modelo final y no las tablas intermedias de cada paso.

In [13]:
modelo_backward <- step(modelo_completo, direction = "backward", trace = FALSE)

formula(modelo_backward)
summary(modelo_backward)

Balance ~ Income + Limit + Rating + Cards + Age + Student


Call:
lm(formula = Balance ~ Income + Limit + Rating + Cards + Age + 
    Student, data = Credit)

Residuals:
    Min      1Q  Median      3Q     Max 
-170.00  -77.85  -11.84   56.87  313.52 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -493.73419   24.82476 -19.889  < 2e-16 ***
Income        -7.79508    0.23342 -33.395  < 2e-16 ***
Limit          0.19369    0.03238   5.981 4.98e-09 ***
Rating         1.09119    0.48480   2.251   0.0250 *  
Cards         18.21190    4.31865   4.217 3.08e-05 ***
Age           -0.62406    0.29182  -2.139   0.0331 *  
StudentYes   425.60994   16.50956  25.780  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 98.61 on 393 degrees of freedom
Multiple R-squared:  0.9547,	Adjusted R-squared:  0.954 
F-statistic:  1380 on 6 and 393 DF,  p-value: < 2.2e-16


In [14]:
data.frame(
  Modelo = c("Problema 1", "Completo", "Backward AIC"),
  R2_aj  = c(summary(modelo1)$adj.r.squared,
             summary(modelo_completo)$adj.r.squared,
             summary(modelo_backward)$adj.r.squared),
  AIC    = c(AIC(modelo1), AIC(modelo_completo), AIC(modelo_backward))
)

Modelo,R2_aj,AIC
<chr>,<dbl>,<dbl>
Problema 1,0.8747894,5215.578
Completo,0.9538287,4823.370
Backward AIC,0.9539961,4817.039


In [15]:
anova(modelo_backward, modelo_completo)

,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,393,3821620,NA,NA,NA,NA
2,388,3786730,5,34889.48,0.7149766,0.612486


El procedimiento backward seleccionó el modelo$$Balance \sim Income + Limit + Rating + Cards + Age + Student$$eliminando `Region`, `Married`, `Education` y `Own`. Respecto del modelo completo, el $R^2_{aj}$ se mantiene prácticamente igual (0,9540 frente a 0,9538) mientras el AIC disminuye de 4823,4 a 4817,0: se obtiene el mismo poder explicativo con cuatro predictores menos.Las variables descartadas son las demográficas y geográficas, que no aportan información adicional una vez controladas las variables financieras. El test F entre ambos modelos lo confirma: $F(5, 388) = 0{,}715$ con $p = 0{,}61$, es decir, no se rechaza que los coeficientes eliminados sean todos nulos.Cabe notar que el criterio AIC mantiene simultáneamente a `Limit` y `Rating` pese a su altísima correlación: el AIC busca ajuste, no estabilidad de los coeficientes, y por eso no resuelve el problema de multicolinealidad que se analiza a continuación.

# Problema 3: Multicolinealidad

### 3.1 Matriz de correlaciones entre predictores numéricos

In [16]:
num <- Credit[, c("Income", "Limit", "Rating", "Cards", "Age", "Education")]
round(cor(num), 3)

,Income,Limit,Rating,Cards,Age,Education
Income,1.000,0.792,0.791,-0.018,0.175,-0.028
Limit,0.792,1.000,0.997,0.010,0.101,-0.024
Rating,0.791,0.997,1.000,0.053,0.103,-0.030
Cards,-0.018,0.010,0.053,1.000,0.043,-0.051
Age,0.175,0.101,0.103,0.043,1.000,0.004
Education,-0.028,-0.024,-0.030,-0.051,0.004,1.000


**¿Qué pares de predictores presentan las correlaciones más altas?**Las correlaciones más altas se dan entre `Limit` y `Rating` (**r = 0,997**), un valor casi perfecto que indica que ambas variables entregan prácticamente la misma información sobre el cliente: el rating crediticio se construye en gran medida a partir del cupo otorgado. Le siguen `Income`–`Limit` (r = 0,792) e `Income`–`Rating` (r = 0,791).El resto de los predictores (`Cards`, `Age`, `Education`) presenta correlaciones cercanas a cero entre sí y con las demás variables (todas bajo 0,18 en valor absoluto).

### 3.2 Factor de inflación de varianza (VIF)Modelo considerado: `Balance ~ Income + Limit + Rating + Cards + Age + Education`.$VIF_j = \dfrac{1}{1 - R_j^2}$, donde $R_j^2$ es el $R^2$ de regresionar el predictor $j$ contra los demás predictores.

In [17]:
predictores <- c("Income", "Limit", "Rating", "Cards", "Age", "Education")

vif_manual <- sapply(predictores, function(v) {
  f <- as.formula(paste(v, "~", paste(setdiff(predictores, v), collapse = " + ")))
  1 / (1 - summary(lm(f, data = Credit))$r.squared)
})

round(vif_manual, 3)

Income     Limit    Rating     Cards       Age Education 
    2.773   228.848   230.613     1.434     1.039     1.008

**¿Qué variables presentan los valores más altos de VIF? ¿Existe evidencia de problemas de multicolinealidad?**Los VIF obtenidos son: `Income` 2,77, `Limit` 228,85, `Rating` 230,61, `Cards` 1,43, `Age` 1,04 y `Education` 1,01.`Rating` y `Limit` presentan valores extremadamente altos, muy por encima del umbral habitual de 10, por lo que **existe evidencia clara de multicolinealidad severa**. Sus varianzas están infladas más de 200 veces respecto de un escenario sin correlación, lo que equivale a errores estándar unas $\sqrt{230} \approx 15$ veces más grandes. El problema se origina casi por completo en la altísima correlación entre `Limit` y `Rating`. `Income` muestra un VIF moderado (2,77), aceptable, mientras que `Cards`, `Age` y `Education` no presentan indicios de multicolinealidad.El efecto es directamente visible en el coeficiente de `Limit`:

In [18]:
modelo_vif <- lm(Balance ~ Income + Limit + Rating + Cards + Age + Education,
                 data = Credit)

rbind(con_Rating = round(summary(modelo_vif)$coefficients["Limit", ], 4),
      sin_Rating = round(summary(modelo1)$coefficients["Limit", ], 4))

,Estimate,Std. Error,t value,Pr(>|t|)
con_Rating,0.1259,0.0530,2.3729,0.0181
sin_Rating,0.2628,0.0058,45.3287,0.0000


Al incluir `Rating`, el coeficiente de `Limit` cae de 0,263 a 0,126 y su error estándar se multiplica por 9 (de 0,0058 a 0,0530), al punto de dejar de ser significativo. El efecto no desapareció: simplemente se repartió entre dos variables que miden lo mismo.

### 3.3 ¿Por qué la colinealidad dificulta la interpretación de los coeficientes?

El coeficiente $\beta_j$ mide el efecto de $X_j$ **manteniendo constantes los demás predictores**. Si dos predictores se mueven casi juntos (r = 0,997), en los datos casi no existen observaciones donde uno cambie y el otro no, de modo que ese "manteniendo constante" no está respaldado por información real: los datos no permiten separar su efecto individual.En términos algebraicos, $X'X$ queda cerca de ser singular y $(X'X)^{-1}$ tiene entradas muy grandes; como $\widehat{Var}(\hat\beta) = \hat\sigma^2 (X'X)^{-1}$, la varianza de los $\hat\beta$ se infla. Las consecuencias prácticas son:- los **errores estándar crecen** y los estadísticos $t$ caen, produciendo falsos "no significativos": el efecto conjunto es fuerte pero ninguno de los dos predictores resulta significativo por separado;- los coeficientes se vuelven **inestables**: cambian mucho de magnitud e incluso de signo al agregar o quitar una variable, o al modificar unas pocas observaciones;- todo esto ocurre **incluso con $R^2$ alto**: la colinealidad daña la interpretación de los coeficientes individuales, no la capacidad predictiva del modelo.

# Problema 4: Predicción y regularización

### 4.1 Investigue brevemente en qué consiste la validación cruzada K-fold y explique con sus propias palabras cómo permite estimar el error predictivo.Los datos se dividen al azar en $K$ subconjuntos (*folds*) de tamaño similar. El procedimiento se repite $K$ veces: en la iteración $k$ se ajusta el modelo usando los $K-1$ folds restantes (**entrenamiento**) y se predicen las observaciones del fold $k$ (**validación**), que no participaron del ajuste. Se calcula el error en ese fold y al final se promedian los $K$ errores.La clave es que **cada observación se predice exactamente una vez con un modelo que no la vio**. Eso evita el optimismo del error de entrenamiento —que siempre baja al agregar predictores— y entrega una estimación honesta del error que cometeríamos con un individuo nuevo. Comparado con partir en una sola muestra de entrenamiento y prueba, K-fold usa todos los datos para entrenar y para validar, lo que reduce la varianza de la estimación. Además, todo el proceso de modelamiento (incluida la selección de variables) debe ejecutarse **dentro** de cada fold; si se selecciona con todos los datos y luego se valida, el error resulta artificialmente bajo.

**Partición común a los tres procedimientos**Se fija la semilla y se construye una única partición en $K = 5$ folds, que se reutiliza en 4.2, 4.3 y 4.4 para que la comparación sea justa.

In [19]:
set.seed(123)
K <- 5
folds <- sample(rep(1:K, length.out = nrow(Credit)))

table(folds)

rmse <- function(y, y_hat) sqrt(mean((y - y_hat)^2))

folds
 1  2  3  4  5 
80 80 80 80 80 

### 4.2 Validación cruzada del modelo lineal completoEn cada fold se ajusta `Balance ~ .` con las observaciones de entrenamiento y se calcula$$RMSE_k = \sqrt{\frac{1}{n_k} \sum_{i \in I_k} \left(Y_i - \hat{Y}_i\right)^2}$$

In [20]:
rmse_lineal <- numeric(K)

for (k in 1:K) {
  entrena <- Credit[folds != k, ]
  valida  <- Credit[folds == k, ]

  ajuste <- lm(Balance ~ ., data = entrena)
  pred   <- predict(ajuste, newdata = valida)

  rmse_lineal[k] <- rmse(valida$Balance, pred)
}

round(rmse_lineal, 2)
mean(rmse_lineal)

[1] 101.25  89.70 101.34  96.38 113.52

[1] 100.4364

**RMSE promedio = 100,44 dólares.** En promedio, el modelo se equivoca en unos 100 dólares al predecir el saldo de un individuo que no usó para ajustarse. Como referencia, la desviación estándar de `Balance` es 460, de modo que el modelo reduce el error a menos de un cuarto de la variabilidad original.

### 4.3 Validación cruzada con selección de variables mediante AICLa selección backward se ejecuta **dentro de cada fold**, usando únicamente las observaciones de entrenamiento. Esto es esencial: si se seleccionaran las variables con los 400 datos y luego se validara, el fold de validación habría influido en qué variables entraron al modelo y el RMSE resultaría artificialmente bajo.

In [21]:
rmse_aic  <- numeric(K)
variables <- character(K)

for (k in 1:K) {
  entrena <- Credit[folds != k, ]
  valida  <- Credit[folds == k, ]

  completo <- lm(Balance ~ ., data = entrena)
  elegido  <- step(completo, direction = "backward", trace = FALSE)

  variables[k] <- paste(names(coef(elegido))[-1], collapse = ", ")
  pred <- predict(elegido, newdata = valida)

  rmse_aic[k] <- rmse(valida$Balance, pred)
}

round(rmse_aic, 2)
mean(rmse_aic)

[1] 100.48  89.11 102.84  95.99 111.60

[1] 100.0026

In [22]:
data.frame(fold = 1:K, variables)

fold,variables
<int>,<chr>
1,"Income, Limit, Rating, Cards, Age, StudentYes"
2,"Income, Limit, Rating, Cards, Age, StudentYes"
3,"Income, Limit, Rating, Cards, StudentYes"
4,"Income, Limit, Rating, Cards, Age, OwnYes, StudentYes"
5,"Income, Limit, Rating, Cards, Age, StudentYes"


**RMSE promedio = 100,00 dólares.** La selección resulta muy estable entre folds: `Income`, `Limit`, `Rating`, `Cards` y `Student` aparecen en los cinco, `Age` en cuatro de cinco y `Own` entra solo una vez (fold 4). Esto coincide con el modelo elegido en el Problema 2 y sugiere que esas variables capturan efectivamente la señal, y no un artefacto de la muestra particular.

### 4.4 Regresión Ridge

**Derivación**Ridge minimiza la suma de cuadrados **penalizada** por el tamaño de los coeficientes:$$Q(\beta) = (y - Z\beta)'(y - Z\beta) + \lambda\,\beta'\beta, \qquad \lambda \ge 0$$donde $Z$ contiene los predictores **estandarizados** e $y$ está centrada. El intercepto no se penaliza, porque de lo contrario la solución dependería de las unidades de medida de las variables.Derivando respecto de $\beta$ e igualando a cero:$$\frac{\partial Q}{\partial \beta} = -2Z'(y - Z\beta) + 2\lambda\beta = 0\;\Longrightarrow\; (Z'Z + \lambda I)\,\hat\beta = Z'y$$$$\hat\beta_{ridge} = (Z'Z + \lambda I)^{-1} Z'y$$Sumar $\lambda I$ a la diagonal vuelve la matriz **siempre invertible**, incluso cuando $Z'Z$ es casi singular por multicolinealidad: exactamente el problema del par `Limit`–`Rating` detectado en el Problema 3. El precio es que $\hat\beta_{ridge}$ es **sesgado**, ya que encoge los coeficientes hacia cero. Con $\lambda = 0$ se recupera mínimos cuadrados; con $\lambda \to \infty$ todos los coeficientes tienden a cero.**¿Qué representa $\lambda_{mín}$?** Es el valor de la penalización que **minimiza el error de validación cruzada interna** que realiza `cv.glmnet` sobre los datos de entrenamiento. Marca el mejor equilibrio entre sesgo y varianza: penalizar poco deja el modelo inestable (mucha varianza), penalizar demasiado lo deja rígido y sesgado.

**Implementación con la librería**En cada uno de los 5 folds: (a) se selecciona $\lambda_{mín}$ con `cv.glmnet()` y $\alpha = 0$ usando solo los datos de entrenamiento, (b) se predice el fold de validación y (c) se calcula el RMSE.

In [23]:
X <- model.matrix(Balance ~ ., data = Credit)[, -1]   # dummies, sin intercepto
Y <- Credit$Balance

rmse_ridge <- numeric(K)
lambdas    <- numeric(K)

set.seed(123)
for (k in 1:K) {
  Xe <- X[folds != k, ];  ye <- Y[folds != k]   # entrenamiento
  Xv <- X[folds == k, ];  yv <- Y[folds == k]   # validación

  cv <- cv.glmnet(Xe, ye, alpha = 0)            # alpha = 0  ->  ridge
  lambdas[k] <- cv$lambda.min

  pred <- predict(cv, newx = Xv, s = "lambda.min")
  rmse_ridge[k] <- rmse(yv, pred)
}

round(rmse_ridge, 2)
mean(rmse_ridge)
round(lambdas, 2)

[1] 118.09 136.30 119.70 110.71 109.71

[1] 118.8996

[1] 40.42 39.36 40.13 39.79 38.57

**RMSE promedio = 118,90 dólares**, con un $\lambda_{mín}$ muy estable entre folds (entre 38,6 y 40,4).

**Verificación con la fórmula cerrada**Para comprobar la derivación, aplicamos $\hat\beta = (Z'Z + \lambda I)^{-1}Z'y$ **una sola vez**, sobre el fold 1, reutilizando el `lambda.min` que `cv.glmnet` ya seleccionó arriba.Nota: `glmnet` minimiza $\frac{1}{2n}\|y - Z\beta\|^2 + \frac{\lambda}{2}\|\beta\|^2$ y estandariza internamente la respuesta, de modo que su $\lambda$ equivale a $\lambda^{*} = n\,\lambda / s_y$ en la fórmula de arriba.

In [24]:
Xe <- X[folds != 1, ]
ye <- Y[folds != 1]

m  <- colMeans(Xe)
s  <- apply(Xe, 2, function(z) sqrt(mean((z - mean(z))^2)))
Z  <- scale(Xe, center = m, scale = s)      # predictores estandarizados
yc <- ye - mean(ye)                         # respuesta centrada

lambda <- length(ye) * lambdas[1] / sqrt(mean(yc^2))   # lambda.min en la escala de la fórmula

b      <- solve(t(Z) %*% Z + lambda * diag(ncol(Z)), t(Z) %*% yc)
beta   <- as.vector(b) / s                  # vuelta a la escala original
manual <- c("(Intercepto)" = mean(ye) - sum(m * beta), beta)

data.frame(
  manual = round(manual, 4),
  glmnet = round(as.vector(coef(glmnet(Xe, ye, alpha = 0, lambda = lambdas[1]))), 4)
)

,manual,glmnet
,<dbl>,<dbl>
(Intercepto),-419.4170,-419.4808
Income,-5.0580,-5.0613
Limit,0.1141,0.1143
Rating,1.6607,1.6580
Cards,13.7422,13.7542
Age,-0.7611,-0.7609
Education,0.1908,0.1917
OwnYes,-4.5182,-4.5299
StudentYes,372.2585,372.2843


Ambos caminos entregan los mismos coeficientes: la fórmula cerrada resuelve el sistema de forma exacta con `solve()` y `glmnet` llega al mismo punto mediante un algoritmo iterativo, así que las diferencias aparecen recién en la cuarta cifra decimal.

### 4.5 Resumen de resultados y recomendación

In [25]:
data.frame(
  Procedimiento = c("Modelo lineal completo", "Selección mediante AIC", "Regresión Ridge"),
  RMSE_promedio = round(c(mean(rmse_lineal), mean(rmse_aic), mean(rmse_ridge)), 2)
)

Procedimiento,RMSE_promedio
<chr>,<dbl>
Modelo lineal completo,100.44
Selección mediante AIC,100.00
Regresión Ridge,118.90


| Procedimiento | RMSE promedio ||---|---|| Modelo lineal completo | 100,44 || Selección mediante AIC | 100,00 || Regresión Ridge | 118,90 |**¿Qué procedimiento utilizaríamos para predecir el `Balance` de un nuevo individuo?**El **modelo seleccionado mediante AIC**. Obtiene el RMSE más bajo y además usa 6 predictores en lugar de 11, lo que lo hace más simple de comunicar y aplicar. Con todo, hay que ser honestos con la magnitud de la ventaja: la diferencia con el modelo lineal completo es de 0,44 dólares, ruido frente a la variabilidad entre folds.

In [26]:
sd(rmse_lineal)

[1] 8.723552

La desviación estándar del RMSE entre folds es de 8,7 dólares, veinte veces mayor que la diferencia entre ambos procedimientos. En rigor, el modelo completo y el seleccionado por AIC predicen igual de bien; la selección por AIC se prefiere por **parsimonia**, no por precisión.Ridge, en cambio, queda casi 19 dólares atrás. El encogimiento se ve directamente al comparar los coeficientes sobre todos los datos:

In [27]:
set.seed(123)
cv_todo <- cv.glmnet(X, Y, alpha = 0)

round(cbind(Ridge = as.vector(coef(cv_todo, s = "lambda.min")),
            MCO   = coef(modelo_completo)), 3)

,Ridge,MCO
(Intercept),-400.860,-479.208
Income,-5.175,-7.803
Limit,0.114,0.191
Rating,1.661,1.137
Cards,15.806,17.724
Age,-0.957,-0.614
Education,-0.474,-1.099
OwnYes,-4.855,-10.653
StudentYes,381.690,425.747
MarriedYes,-12.098,-8.534


Ridge reduce en valor absoluto casi todos los coeficientes: `Income` pasa de −7,80 a −5,18, `Limit` de 0,191 a 0,114 y `Student` de 425,7 a 381,7. El problema es que esos coeficientes **sí eran útiles**: encogerlos introduce un sesgo que no se compensa con una reducción de varianza, porque con $n = 400$ observaciones frente a 11 predictores y un $R^2 \approx 0{,}95$ los coeficientes de mínimos cuadrados ya son estables.La regularización rinde cuando hay muchos predictores respecto del tamaño muestral o cuando la señal es débil, y aquí ocurre lo contrario. Ridge sí resuelve el problema detectado en el Problema 3 —estabiliza `Limit` y `Rating`—, pero eso mejora la **interpretación** de los coeficientes, no la capacidad predictiva.

# ConclusiónLos predictores relevantes del saldo son `Limit`, `Income`, `Rating`, `Cards`, `Age` y sobre todo `Student` (+426 dólares); `Education`, `Own`, `Married` y `Region` no aportan y son descartadas por AIC. Existe multicolinealidad severa entre `Limit` y `Rating` (r = 0,997; VIF sobre 228), que infla los errores estándar y vuelve inestables ambos coeficientes, aunque no afecta la predicción. En validación cruzada con K = 5, la selección por AIC obtuvo el mejor RMSE (100,0), seguida del modelo completo (100,4) y de Ridge (118,9). Con n = 400 y señal fuerte (R² ≈ 0,95), la regularización no aportó ventaja predictiva.